# 命名实体识别（NER）
1.  **用户输入 Query**：例如，“姚明的身高是多少？”。
2.  **Query 理解**：后台对 Query 进行一系列处理，尝试“读懂”用户的真实意图。
    - **基础处理**：分词、词性标注、纠错等。
    - **深度理解**：
        - **意图识别**：首先判断 Query 属于哪种意图，例如“属性查询”、“汇率转换”、“单位换算”等。
        - **命名实体识别**：根据识别出的意图，有针对性地抽取相应的实体。
            - **属性查询** ("姚明的身高是多少？")：抽取“主体”（姚明）和“属性类型”（身高）。
            - **汇率转换** ("50欧元等于多少日元")：抽取“源汇率金额”(50)、“源汇率类型”(欧元)、“目标汇率类型”(日元)。
            - **单位换算** ("5英尺等于多少厘米")：抽取“源度量值”(5)、“源单位”(英尺)、“目标单位”(厘米)。
3.  **信息检索**：根据理解结果，从庞大的知识图谱或索引库中精准匹配答案。
4.  **结果排序与呈现**：将最相关的结果排序后，以结构化的方式优先呈现给用户。
#### 4.2.1 方案一：Token 级别标签预测 (BMES/BIO)

这种方法为每个 Token 预测其在实体中扮演的角色，是序列标注最经典的思想。

- **标注体系**:
    - **BMES**: `B`(Begin), `M`(Middle), `E`(End), `S`(Single)
    - **BIO**: `B`(Begin), `I`(Inside), `O`(Outside)
- **示例**:
    - **文本 (x)**:       `西 安 的 大 雁 塔 门 票 多 少 钱`
    - **BMES 体系 (y)**: `B-LOC E-LOC O B-LOC M-LOC E-LOC B-ATTR E-ATTR O O O`
    - **BIO 体系 (y)**:  `B-LOC I-LOC O B-LOC I-LOC I-LOC B-ATTR I-ATTR O O O`
    > 上例中，`LOC` 代表“地点”（Location），如“西安”和“大雁塔”；`ATTR` 代表“属性”（Attribute），此处指“门票”。
- **模型结构**: `Token Embedding层 -> 序列模型层 -> Token分类层`
    - **Token Embedding/Encoder 表示**: 可以是静态的 `nn.Embedding`；更常见的是直接使用预训练 Encoder（如 `BERT`/`RoBERTa`/`DeBERTa` 等）的上下文表示。
    - **Token Classify**:
        - **Softmax**: 对每个Token独立分类，类别总数为 `1 (非实体) + 实体类别数 * 4` (以BMES为例)。
        - **CRF**: 在 Softmax 基础上，额外学习标签之间的转移概率（如 B-LOC 后更可能是 M-LOC），对整个序列进行全局最优解码。对 RNN/CNN 等编码器通常有效；但在强大的预训练 Encoder（如 BERT）上收益在不少数据集上已变小，是否采用以实验为准。
- **缺点**: 无法解决实体嵌套问题。例如，对于文本“他就读于北京大学”，这种方法很难同时识别出“北京”（地名）和“北京大学”（组织机构名）这两个存在包含关系的实体。一个更理想的嵌套实体示例如下：
  ```json
  {
      "token": ["他", "就", "读", "于", "北", "京", "大", "学"],
      "span": [
          {"type": "LOC", "start": 4, "end": 5},
          {"type": "ORG", "start": 4, "end": 7}
      ]
  }
  ```
#### 4.2.2 方案二：指针网络与片段网络

这类方法主要是为了解决实体嵌套问题，是当前处理复杂 NER 场景的主流方案之一。

- **片段网络**[^2][^3]:
    - **思路**: “暴力美学”——枚举出文本中所有可能的连续片段，然后用一个分类器去判断每一个片段属于哪个实体类型（或是“非实体”）。
    - **示例**: 对于文本 “美式咖啡”，片段网络会进行如下操作：
        1.  **枚举所有候选片段**:
            -   长度为 1: `美`, `式`, `咖`, `啡`
            -   长度为 2: `美式`, `式咖`, `咖啡`
            -   长度为 3: `美式咖`, `式咖啡`
            -   长度为 4: `美式咖啡`
        2.  **对每个片段进行分类**:
            -   `美式` -> `PROD`
            -   `咖啡` -> `PROD`
            -   `美式咖啡` -> `PROD`
            -   (所有其他片段) -> `O` (非实体)
    - **优点**: 理论上可以解决实体嵌套问题。
    - **缺点**:
        1.  **计算量大**: 候选片段数量随文本长度平方级增长（如上例，4个字的文本就产生了10个候选片段）。
        2.  **样本不均衡**: 绝大多数片段都是非实体，导致正负样本严重失衡，训练困难。
- **指针网络**[^4]:
    - **思路**: 与其为每个 token 打一个固定的 BMES 标签，指针网络的思想是为每个 token 训练多个独立的二分类器，分别判断它是否是“某类实体的开头”以及“某类实体的结尾”。这种方式非常适合作为生成候选片段的第一步。
    - **示例**: 对于句子 “来一杯星巴克的美式咖啡”，如果我们想同时识别出“星巴克”（机构名）以及嵌套的“美式”、“咖啡”、“美式咖啡”（产品名），指针网络的输出会是这样：
      | Token | is_ORG_start | is_ORG_end | is_PROD_start | is_PROD_end | ... |
      | :---: | :----------: | :--------: | :-----------: | :---------: | :-: |
      |   来  |      0       |     0      |       0       |      0      | ... |
      |   一  |      0       |     0      |       0       |      0      | ... |
      |   杯  |      0       |     0      |       0       |      0      | ... |
      |   星  |      1       |     0      |       0       |      0      | ... |
      |   巴  |      0       |     0      |       0       |      0      | ... |
      |   克  |      0       |     1      |       0       |      0      | ... |
      |   的  |      0       |     0      |       0       |      0      | ... |
      |   美  |      0       |     0      |       1       |      0      | ... |
      |   式  |      0       |     0      |       0       |      1      | ... |
      |   咖  |      0       |     0      |       1       |      0      | ... |
      |   啡  |      0       |     0      |       0       |      1      | ... |
    - **候选生成**: 得到预测后，后处理程序会按实体类型分别进行“开头-结尾”配对：
        - **ORG 类型**: `星` (start) + `克` (end)  =>  `星巴克`
        - **PROD 类型**:
            - `美` (start) + `式` (end)  =>  `美式`
            - `咖` (start) + `啡` (end)  =>  `咖啡`
            - `美` (start) + `啡` (end)  =>  `美式咖啡`
    > 通过这种“判断边界，再组合配对”的方式，指针网络巧妙地生成了所有可能的实体片段（包括嵌套的），为后续的分类环节提供了高质量的候选。
- **指针网络 + 片段网络**[^5]:
    - **思路**: 结合两者的长处，形成一个高效的两阶段流程。
    1.  **候选生成 (指针网络)**: 先使用指针网络高效地预测出所有可能的实体“开头”和“结尾”。
    2.  **候选组合**: 将所有合法的“开头-结尾”配对，组合成候选实体片段。这个过程极大地减少了候选片段的数量，过滤掉了绝大多数无意义的组合。
    3.  **候选分类 (片段网络)**: 再使用片段网络对这些数量大大减少的 **候选片段** 进行分类。
    - **优点**: 既能解决嵌套问题，又有效降低了计算量，是解决复杂 NER 问题的有效方案。

#### 4.2.3 方案三：基于分词的分类

- **思路**: 将 NER 任务分解为两步：先调用一个现成的分词模型将文本切分成词语，然后对每个 **词语** 进行分类。
- **示例**:
    - **原始文本**: `西安的大雁塔门票多少钱`
    - **分词结果**: `西安`, `的`, `大雁塔`, `门票`, `多少`, `钱`
    - **分类结果**: `LOC`, `O`, `LOC`, `ATTR`, `O`, `O`
- **缺点**: 效果高度依赖上游分词模型的质量。如果分词出错（例如，将“大雁塔”错分成“大雁”和“塔”），实体识别基本不可能正确，可谓“一步错，步步错”。

In [7]:
import json
import os
from collections import Counter
from torch.utils.data import Dataset,DataLoader
from torch.nn.utils.rnn import pad_sequence
import torch

def save_json(data, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

def collect_entity_types_from_file(file_path):
    with open(file_path,'r',encoding='utf-8') as f:
        all_data = json.load(f)
    types = set()
    for item in all_data:
        for entity in item['entities']:
            types.add(entity['type'])
    return types
   
def generate_tag_map(file_name,output_file):
    all_types = set()
    for file in file_name:
        types = collect_entity_types_from_file(file)
        all_types.update(types)
    sorted_types = sorted(list(all_types))    
    tag_to_id = {"O": 0}
    for type in sorted_types:
        for prefix in ["B",'M','E','S']:
            tag_name = f"{prefix}-{type}"
            tag_to_id[tag_name] = len(tag_to_id)
    print(len(tag_to_id))
    save_json(tag_to_id, output_file)

def normalize_text(text):
    full_width = "０１２３４５６７８９ＡＢＣＤＥＦＧＨＩＪＫＬＭＮＯＰＱＲＳＴＵＶＷＸＹＺａｂｃｄｅｆｇｈｉｊｋｌｍｎｏｐｑｒｓｔｕｖｗｘｙｚ！＃＄％＆’（）＊＋，－．／：；＜＝＞？＠［＼］＾＿｀｛｜｝～＂"
    half_width = r"0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz!#$%&'" + r'()*+,-./:;<=>?@[\]^_`{|}~".'
    mapping = str.maketrans(full_width, half_width) ## 全角转半角的映射表
    return text.translate(mapping)

def create_char_vocab(data_file,output_file,min_freq=1):
    with open(data_file,'r',encoding='utf-8') as f:
        all_data = json.load(f)
    char_counter = Counter()
    for item in all_data:
        for text in item['text']:
            normalized_text = normalize_text(text)
            char_counter.update(normalized_text)  # list(char,count)
    frequent_chars = {char for char, count in char_counter.items() if count >= min_freq}
    special_tokens = ["<PAD>", "<UNK>"]
    final_vocab = special_tokens + sorted(frequent_chars)
    save_json(final_vocab, output_file)

class Vocabulary:
    def __init__(self,vocab_file):
        with open(vocab_file,'r',encoding='utf-8') as f:
            self.vocab = json.load(f)
        self.token_to_ids = {token: idx for idx, token in enumerate(self.vocab)}
        self.pad_id = self.token_to_ids.get("<PAD>")
        self.unk_id = self.token_to_ids.get("<UNK>")
    def __len__(self):
        return len(self.vocab)
    def token_to_id(self,tokens):
        return (self.token_to_ids.get(token , self.unk_id) for token in tokens)
    
class NERDataset(Dataset):
    def __init__(self,data_file, vocab, tag_map):
        with open(data_file,'r',encoding='utf-8') as f:
            self.data = json.load(f)
        self.vocab = vocab
        self.tag_map = tag_map
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        text = item['text']
        normalized_text = normalize_text(text)
        text_ids = list(self.vocab.token_to_id(normalized_text))
        tags = ["O"] * len(text_ids)
        for entity in item['entities']:
            start,end,entity_type = entity['start_idx'],entity['end_idx'],entity['type']
            if end > len(tags):
                continue
            if start == end:
                tags[start] = f"S-{entity_type}"
            else:
                tags[start] = f"B-{entity_type}"
                tags[end] = f"E-{entity_type}"
                for i in range(start+1,end):
                    tags[i] = f"M-{entity_type}"
        label_ids = [self.tag_map[tag] for tag in tags]
        return{
            "token_ids": torch.tensor(text_ids, dtype=torch.long),
            "label_ids": torch.tensor(label_ids, dtype=torch.long)
        }

def create_ner_dataloader(data_path,vocab,tag_map,batch_size=32,shuffle=False):
    dataset = NERDataset(data_path,vocab,tag_map)
    def collate_fn(batch):
        token_ids = [item['token_ids'] for item in batch]
        label_ids = [item['label_ids'] for item in batch]
        padded_token_ids = pad_sequence(token_ids, batch_first=True, padding_value=vocab.pad_id)
        padded_label_ids = pad_sequence(label_ids, batch_first=True, padding_value=-100)##-100为ignore_index
        attention_mask = (padded_token_ids != vocab.pad_id).long()
        return{
            "token_ids": padded_token_ids,
            "label_ids": padded_label_ids,
            "attention_mask": attention_mask
        }
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_fn)

if __name__ == "__main__":
    train_file = "./data/CMeEE-V2_train.json"
    dev_file = "./data/CMeEE-V2_dev.json"
    out_put_file = "./data/categories.json"
    output_path = "./data/vocabulary.json"
    generate_tag_map(file_name=[train_file, dev_file], output_file=out_put_file)
    create_char_vocab(train_file, output_path, min_freq=1)
    vocab = Vocabulary(output_path)
    tag_map = json.load(open(out_put_file, 'r', encoding='utf-8'))
    dataset = NERDataset(train_file, vocab, tag_map)
    train_loader = create_ner_dataloader(train_file, vocab, tag_map, batch_size=4, shuffle=True)
    batch = next(iter(train_loader))
    print(batch)
    print("\n--- DataLoader 输出验证 ---")
    print(f"  Token IDs shape: {batch['token_ids'].shape}")
    print(f"  Label IDs shape: {batch['label_ids'].shape}")
    print(f"  Attention Mask shape: {batch['attention_mask'].shape}")

37
{'token_ids': tensor([[1688,  201,  145,  206, 2140, 1725, 2205,    9,  723, 2063, 1582, 2608,
         1662, 2205,  554, 1482, 1012,  611, 2304, 1275, 2205,  499,  234, 1228,
         1389,  878,  489, 2115, 2205, 2018, 1763,  145, 2542,  377,    9,  226,
         1738, 1320, 1267,  377, 1860,  374, 1763, 1725,  470, 2205,  152,  145,
          775, 1228, 2140, 1725, 1763,  470,  636,   24, 2028,  156, 2708,  340,
          660, 1725,  470,  169,  723, 2063, 1582, 1173,  470,  269,  138, 2063,
         1582, 2309,  470,  269,  234,  478,  467, 1420, 2205,    9,  878, 2339,
         1738,  668,  341, 2686,  234, 2202,  950, 2658,  980, 1979, 1300,    9,
          636, 2028, 1738,  668,  341, 1763, 1979, 1300,  191,  495, 2091,  273,
          175,  172,  893, 1451, 1708, 1763,  474, 2026,  139,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0],
        [2063, 2304, 1482, 2574, 1763,  

# 构建GRU模型
> **nn.ModuleList vs nn.Sequential**
> 
> 在 PyTorch 中，`nn.ModuleList` 和 `nn.Sequential` 都是用来容纳多个子模块的容器，但它们的设计思想和使用场景不同：
> - **`nn.Sequential`**：像一个自动化的**流水线**，数据会自动按顺序流过每一层。适用于简单的线性堆叠，但无法实现层间的复杂交互。
> - **`nn.ModuleList`**：更像一个普通的 **Python 列表**，只负责存储模块，而不会自动执行它们。你需要在 `forward` 方法中手动编写循环来调用每一层，所以可以在层与层之间加入自定义逻辑（如残差连接）。

In [8]:
import torch
import torch.nn as nn
import torch.nn.utils.rnn as rnn

class GRUNerNetWork(nn.Module):
    def __init__(self,vocab_size,hidden_size,num_tags,gru_layers=1):
        super(GRUNerNetWork,self).__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.gru = nn.ModuleList()
        for i in range(gru_layers):
            self.gru.append(nn.GRU(input_size=hidden_size, hidden_size=hidden_size, num_layers=1,batch_first=True,bidirectional=False))
        self.classifier = nn.Linear(hidden_size, num_tags)
    def forward(self, token_ids, attention_mask=None):
        embeddings = self.embedding(token_ids)
        input = embeddings
        for gru_layer in self.gru:
            gru_output, _ = gru_layer(input)
            input = gru_output + input
        logits = self.classifier(input)
        return logits

if __name__ == '__main__':
    
    token_ids = torch.tensor([
        [210,   18,  871, 147,   0,   0,   0,   0], 
        [922, 2962,  842, 210,  18, 871, 147,   0]
    ], dtype=torch.int64)
    
    # attention_mask 标记哪些是真实 token (1) 哪些是填充 (0)
    attention_mask = torch.tensor([
        [1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 0]
    ], dtype=torch.int64)
    
    label_ids = torch.tensor([
        [0, 0, 0, 0, -100, -100, -100, -100],
        [0, 0, 0, 0,    0,    0,    0, -100]
    ], dtype=torch.int64)
    
    model = GRUNerNetWork(vocab_size=10000, hidden_size=128, num_tags=37, gru_layers=2)
    logits = model(token_ids)## (batch_size, seq_len, num_tags)
    loss_fn = nn.CrossEntropyLoss(ignore_index=-100,reduction="none")##(batch_size,num_tags,seq_len)
    permuted_logits = logits.permute(0, 2, 1)## (batch_size,num_tags,seq_len)
    loss = loss_fn(permuted_logits, label_ids)## (batch_size, seq_len)
    print(f"Logits shape: {logits.shape}")
    print(f"Loss shape: {loss.shape}")
    print("\n每个 Token 的损失:")
    print(loss)
    

Logits shape: torch.Size([2, 8, 37])
Loss shape: torch.Size([2, 8])

每个 Token 的损失:
tensor([[3.6329, 3.1935, 4.3254, 3.2577, 0.0000, 0.0000, 0.0000, 0.0000],
        [4.4603, 3.2113, 4.2203, 3.5163, 3.0812, 4.2307, 3.2017, 0.0000]],
       grad_fn=<ViewBackward0>)


# 采用双向GRU
`torch.nn.utils.rnn.pack_padded_sequence`:这个函数的主要作用是接收一个 **填充后** 的 `input` 张量，以及一个记录了 **真实长度** 的 `lengths` 列表。它会返回一个 `PackedSequence` 对象，可以把它想象成一个“压缩”后的数据包，其中所有的填充位都被暂时移除了。RNN 模块在接收到这个特殊对象后，其内部就能正确、高效地处理变长序列。

当然，有“打包”就有“解包”。与之对应的 `pad_packed_sequence` 函数会负责将 RNN 计算完成后的 `PackedSequence` 对象再“解压”还原成带有填充的、规整的 Tensor。

### 4.1 核心策略

#### 4.1.1 加权交叉熵损失

最简单的方法就是“加权”。给数量稀少的实体标签（B, M, E, S）一个更高的权重，给数量庞大的非实体标签（O）一个较低的权重。例如，我们可以设置实体损失的权重为 10，非实体损失的权重为 1。这样，模型在反向传播时，如果弄错了一个实体 Token，会受到比弄错一个非实体 Token 大 10 倍的“惩罚”，从而迫使模型更加关注对实体的识别。

#### 4.1.2 硬负样本挖掘

另一种思路是“采样”。在大量的非实体样本中，大部分是模型可以轻易正确预测的“简单样本”，它们对损失的贡献很小，反复学习意义不大。真正有价值的是那些模型容易搞错的“硬负样本”，例如一个模型倾向于预测为实体的非实体 Token。

硬负样本挖掘的做法是：在计算非实体部分的损失时，不计算所有非实体 Token 的平均损失，而是只选择其中损失值最大（Top-K）的一部分进行计算和反向传播。这样就相当于从海量的“多数派”中，筛选出了最有价值的“疑难样本”进行学习，提升了训练的效率和效果。

In [12]:
import torch
import torch.nn as nn
import torch.nn.utils.rnn as rnn
from dataclasses import dataclass,field
import os
import json
import tqdm
from abc import ABC, abstractmethod
from torch.utils.data import Dataset,DataLoader
from torch.nn.utils.rnn import pad_sequence
from dataclasses import asdict
from torch.utils.tensorboard import SummaryWriter
import numpy as np


class EarlyStopping:
    def __init__(self, patience=5, verbose=False, delta=0, monitor="f1", mode="max"):
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.monitor = monitor
        self.mode = mode
        self.val_metric_best = np.inf if mode == "min" else -np.inf
    def __call__(self, val_meitric):
        score = -val_meitric if self.mode == "min" else val_meitric
        if self.best_score is None:
            self.best_score = score
        elif score < self.best_score + self.delta:
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
                self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            best_metric = val_meitric
            self.counter = 0
        return self.early_stop
    def state_dict(self):
        return {
            'patience': self.patience,
            'verbose': self.verbose,
            'delta': self.delta,
            'counter': self.counter,
            'best_score': self.best_score,
            'early_stop': self.early_stop,
            'monitor': self.monitor,
            'mode': self.mode,
            'val_metric_best': self.val_metric_best,
        }

    def load_state_dict(self, state_dict):
        """从字典恢复状态"""
        self.patience = state_dict['patience']
        self.verbose = state_dict['verbose']
        self.delta = state_dict['delta']
        self.counter = state_dict['counter']
        self.best_score = state_dict['best_score']
        self.early_stop = state_dict['early_stop']
        self.monitor = state_dict['monitor']
        self.mode = state_dict['mode']
        self.val_metric_best = state_dict['val_metric_best']
    
class BiGRUNerNetWork(nn.Module):
    def __init__(self,vocab_size,hidden_size,num_tags,gru_layers=1):
        super(BiGRUNerNetWork,self).__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.gru = nn.ModuleList()
        for i in range(gru_layers):
            self.gru.append(nn.GRU(input_size=hidden_size, hidden_size=hidden_size, num_layers=1,batch_first=True,bidirectional=True))
        self.fc = nn.Linear(hidden_size * 2, hidden_size)
        self.classifier = nn.Linear(hidden_size, num_tags)
    def forward(self, token_ids, attention_mask):
        embeddings = self.embedding(token_ids)
        lengths = attention_mask.sum(dim=1).cpu()
        packed_input = rnn.pack_padded_sequence(embeddings, lengths, batch_first=True, enforce_sorted=False)
        input = packed_input
        for gru_layer in self.gru:
            gru_output, _ = gru_layer(input)
            output,_ = rnn.pad_packed_sequence(gru_output,batch_first=True,total_length=token_ids.shape[1])##(batch_size, seq_len, hidden_size*2)
            feature = self.fc(output)
            pad_input,_ = rnn.pad_packed_sequence(input,batch_first=True,total_length=token_ids.shape[1])
            input = feature + pad_input
            input = rnn.pack_padded_sequence(input, lengths, batch_first=True, enforce_sorted=False)
        logits = self.classifier(feature)
        return logits

class TensorBoardLogger:
    def __init__(self, log_dir):
        self.writer = SummaryWriter(log_dir)
    def _log_metrics(self, metrics, step, prefix):
        if isinstance(metrics, tuple):
            self.writer.add_scalar(f"{prefix}/total_loss", metrics[0], step)
            if len(metrics) > 1:
                self.writer.add_scalar(f"{prefix}/ner_loss", metrics[1], step)
            if len(metrics) > 2:
                self.writer.add_scalar(f"{prefix}/non_ner_loss", metrics[2], step)
        elif isinstance(metrics, dict):
            for key, value in metrics.items():
                self.writer.add_scalar(f"{prefix}/{key}", value, step)
    def close(self):
        if self.writer:
            self.writer.close()
            
   
class Trainer:
    def __init__(self,model,optimizer,loss_fn,train_loader,dev_loader=None,eval_metric=None,output_dir=None, summary_log_dir="./model", early_stopping_patience=5,resume_checkpoint=None, device="cpu"):
        self.model = model.to(device)
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.train_loader = train_loader
        self.dev_loader = dev_loader
        self.eval_metric = eval_metric
        self.device = device
        self.start_epoch = 1
        self.best_metric = float('-inf')
        self.output_dir = output_dir
        if self.output_dir:
            os.makedirs(self.output_dir, exist_ok=True)
        self.logger = TensorBoardLogger(summary_log_dir)
        self.early_stopping = EarlyStopping(patience=early_stopping_patience, verbose=True, monitor="f1", mode="max")
        if resume_checkpoint:
            self._resume_checkpoint(resume_checkpoint)
            
    def _resume_checkpoint(self, checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        self.start_epoch = checkpoint['epoch'] + 1
        self.best_metric = checkpoint.get('best_metric', float('-inf'))
        print(f"Resumed training from checkpoint: {checkpoint_path} at epoch {self.start_epoch} with best metric {self.best_metric:.4f}")
    
    def fit(self,epochs):## 训练主入口
        os.makedirs(self.output_dir, exist_ok=True)
        best_metric = float('-inf')
        for epoch in range(self.start_epoch, epochs + 1):
            train_loss = self.train_one_step()
            if isinstance(train_loss, tuple):
                train_loss_str = (
                    f"Train Total Loss: {train_loss[0]:.4f}, "
                    f"NER Loss: {train_loss[1]:.4f}, "
                    f"Non-NER Loss: {train_loss[2]:.4f}"
                )
            else:
                train_loss_str = f"Train Total Loss: {train_loss:.4f}"
            print(train_loss_str)
            eval_metrics = self.evaluate()
            eval_metrics_str = ", ".join([f"{k}: {v:.4f}" for k, v in eval_metrics.items()])
            print(f"Validation Metrics: {eval_metrics_str}")
            is_best = False
            if "f1" in eval_metrics:
                is_best = eval_metrics["f1"] > self.best_metric
                self.best_metric = max(self.best_metric, eval_metrics["f1"])
            else :
                if self.best_metric == float('-inf'):
                    self.best_metric = float('inf')
                else:
                    if eval_metrics["loss"] < self.best_metric:
                        self.best_metric = eval_metrics["loss"]
                        is_best = True
                        
            if is_best:
                print("New best model found! Saving model...")
                self.save_model(is_best=is_best)
            self.save_checkpoint(epoch,is_best)
            self.logger._log_metrics(eval_metrics, epoch, "validation")
            self.logger._log_metrics(train_loss, epoch, "train")
            if self.early_stopping(eval_metrics.get("f1", float('-inf'))):
                print("Early stopping triggered. Stopping training.")
                break
    
    def save_checkpoint(self, epoch, is_best):
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'best_metric': self.best_metric,
            'early_stopping': self.early_stopping.state_dict()
        }
        torch.save(checkpoint, os.path.join(self.output_dir, f'checkpoint_epoch_{epoch}.pth'))
        if is_best:
            torch.save(checkpoint, os.path.join(self.output_dir, 'best_checkpoint.pth'))
    
    def train_one_step(self):
        self.model.train()
        total_loss_sum = 0.0
        total_ner_loss = 0.0
        total_non_ner_loss = 0.0
        custom_loss_used = False
        for batch in tqdm.tqdm(self.train_loader, desc="Training"):
            outputs = self.train_step(batch)
            loss = outputs['loss']
            if isinstance(loss, tuple):
                # 支持元组损失（总损/实体/非实体）并分别累计
                custom_loss_used = True
                total_loss_sum += loss[0].item()
                total_ner_loss += loss[1].item()
                total_non_ner_loss += loss[2].item()
            else:
                total_loss_sum += loss.item()
        if custom_loss_used:
            return (total_loss_sum / len(self.train_loader), total_ner_loss / len(self.train_loader), total_non_ner_loss / len(self.train_loader))
        else:
            return total_loss_sum / len(self.train_loader)
        
    def train_step(self,batch):
        batch = {key: value.to(self.device) for key, value in batch.items()}
        logits = self.model(batch['token_ids'], batch['attention_mask'])
        permuted_logits = logits.permute(0, 2, 1)
        loss = self.loss_fn(permuted_logits, batch['label_ids'])
        main_loss = loss[0] if isinstance(loss, tuple) else loss
        self.optimizer.zero_grad()
        main_loss.backward()
        self.optimizer.step()
        return {"loss": loss,"logits": logits}
        
    def evaluate(self):
        if not self.dev_loader or not self.eval_metric:
            return None
        self.model.eval()
        total_loss = 0.0
        all_logits = []
        all_labels = []
        all_attention_masks = []
        with torch.no_grad():
            for batch in tqdm.tqdm(self.dev_loader, desc="Evaluating"):
                outputs = self.evaluate_step(batch)
                loss = outputs['loss']
                logits = outputs['logits']
                main_loss = loss[0] if isinstance(loss, tuple) else loss
                total_loss += main_loss.item()
                all_logits.append(logits.cpu())
                all_labels.append(batch['label_ids'].cpu())
                all_attention_masks.append(batch['attention_mask'].cpu())
        metrics = {}
        if self.eval_metric:
            metrics = self.eval_metric(all_logits, all_labels, all_attention_masks)
        metrics['loss'] = total_loss / len(self.dev_loader)
        print(f"Evaluation Metrics: {metrics}")
        return metrics
    
    def evaluate_step(self,batch):
        batch = {key: value.to(self.device) for key, value in batch.items()}
        logits = self.model(batch['token_ids'], batch['attention_mask'])
        permuted_logits = logits.permute(0, 2, 1)
        loss = self.loss_fn(permuted_logits, batch['label_ids'])
        return {"loss": loss,"logits": logits}
    
    def save_model(self, is_best=False):
        state = {'model_state_dict': self.model.state_dict()}
        if is_best:
            # 保存最佳模型
            torch.save(state, os.path.join(self.output_dir, 'best_model.pth'))
        # 保存最新模型
        torch.save(state, os.path.join(self.output_dir, 'last_model.pth'))
        
def seed_everything(seed: int = 42):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
        
class NERLoss(nn.Module):
    def __init__(self, loss_type="cross_entropy", entity_weight=10.0, hard_negative_ratio=0.5, ignore_index=-100):
        super(NERLoss, self).__init__()
        self.loss_type = loss_type
        self.entity_weight = entity_weight
        self.hard_negative_ratio = hard_negative_ratio
        self.ignore_index = ignore_index
        self.base_loss_fn = nn.CrossEntropyLoss(ignore_index=ignore_index, reduction="none")
    def forward(self, logits, label_ids):
        if self.loss_type == 'weighted_ce':
            return self._weighted_cross_entropy(logits, label_ids)
        elif self.loss_type == 'hard_negative_mining':
            return self._hard_negative_mining(logits, label_ids)
        else: 
            # 默认使用 PyTorch 原生的交叉熵损失
            return self.base_loss_fn(logits, label_ids).mean()
        
    def _weighted_cross_entropy(self,logits, label_ids):
        loss_per_token = self.base_loss_fn(logits, label_ids)
        entity_mask = (label_ids != self.ignore_index) & (label_ids != 0)
        non_entity_mask = (label_ids == 0)
        entity_loss = torch.sum(loss_per_token *  entity_mask.float())/(entity_mask.sum().float() + 1e-8)
        non_entity_loss = torch.sum(loss_per_token * non_entity_mask.float())/(non_entity_mask.sum().float() + 1e-8)
        total_loss = self.entity_weight * entity_loss + non_entity_loss
        return total_loss,entity_loss.detach(),non_entity_loss.detach()
    
    def _hard_negative_mining(self, logits, label_ids):
        loss_per_token = self.base_loss_fn(logits, label_ids)
        entity_mask = (label_ids != self.ignore_index) & (label_ids != 0)
        entity_loss = torch.sum(loss_per_token * entity_mask.float())/(entity_mask.sum().float() + 1e-8)
        non_entity_mask = (label_ids == 0)
        non_entity_loss = torch.sum(loss_per_token * non_entity_mask.float())/(non_entity_mask.sum().float() + 1e-8)
        num_entity = entity_mask.sum().item()
        num_hard_negative = int(num_entity * self.hard_negative_ratio)
        if num_hard_negative == 0:
            num_non_entity = non_entity_mask.sum().item()
            num_hard_negative = max(1, int(num_non_entity * 0.1))
            
        non_entity_loss_flat = non_entity_loss.view(-1)
        num_non_entity = non_entity_mask.sum().item()
        k = min(num_hard_negative, num_non_entity)
        if k == 0:
            hard_loss = torch.tensor(0.0, device=logits.device)
        else:
            hard_negative_loss, _ = torch.topk(non_entity_loss_flat, k)
            hard_loss = hard_negative_loss.mean()
        total_loss = entity_loss*self.entity_weight + hard_loss
        return total_loss, entity_loss.detach(), hard_loss.detach()
        
        
@dataclass      
class NerConfig:
    data_dir: str = "./data"
    train_file: str = field(init=False, default="CMeEE-V2_train.json")
    dev_file: str = field(init=False, default="CMeEE-V2_dev.json")
    tag_map_file: str = field(init=False, default="categories.json")
    vocab_file: str = field(init=False, default="vocabulary.json")
    output_dir: str = "./model"
    batch_size: int = 32
    epochs: int = 20
    learning_rate: float = 1e-3
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    hidden_size: int = 256
    gru_layers: int = 2
    loss_type: str = "weighted_ce"  # 可选 "cross_entropy", "weighted_ce", "hard_negative_mining"
    entity_weight: float = 10.0
    hard_negative_ratio: float = 0.5
    summary_log_dir: str = "./model"
    seed: int = 42
    early_stopping_patience: int = 5
    resume_checkpoint: str = None

def load_json(file_path):
    """从 JSON 文件加载数据。"""
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def save_json(data, file_path):
    """将数据保存为 JSON 文件。"""
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)



class NERDataset(Dataset):
    def __init__(self,data_file, tokenizer, tag_map):
        with open(data_file,'r',encoding='utf-8') as f:
            self.data = json.load(f)
        self.tokenizer = tokenizer
        self.tag_map = tag_map
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        text = item['text']
        tokens = self.tokenizer.text_to_tokens(text)
        token_ids = self.tokenizer.tokens_to_ids(tokens)
        tags = ["O"] * len(token_ids)
        for entity in item['entities']:
            start,end,entity_type = entity['start_idx'],entity['end_idx'],entity['type']
            if end > len(tags):
                continue
            if start == end:
                tags[start] = f"S-{entity_type}"
            else:
                tags[start] = f"B-{entity_type}"
                tags[end] = f"E-{entity_type}"
                for i in range(start+1,end):
                    tags[i] = f"M-{entity_type}"
        label_ids = [self.tag_map[tag] for tag in tags]
        return{
            "token_ids": torch.tensor(token_ids, dtype=torch.long),
            "label_ids": torch.tensor(label_ids, dtype=torch.long)
        } 
        
def create_ner_dataloader(data_path,tokenizer,tag_map,batch_size=32,shuffle=False):
    dataset = NERDataset(data_path,tokenizer,tag_map)
    def collate_fn(batch):
        token_ids = [item['token_ids'].squeeze() for item in batch]
        label_ids = [item['label_ids'].squeeze() for item in batch]
        padded_token_ids = pad_sequence(token_ids, batch_first=True, padding_value=tokenizer.get_pad_id())
        padded_label_ids = pad_sequence(label_ids, batch_first=True, padding_value=-100)
        attention_mask = (padded_token_ids != tokenizer.get_pad_id()).long()
        return{
            "token_ids": padded_token_ids,
            "label_ids": padded_label_ids,
            "attention_mask": attention_mask
        }
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, collate_fn=collate_fn)

def normalize_text(text):
    half_width = r"0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz!#$%&'" + r'()*+,-./:;<=>?@[\]^_`{|}~".'
    full_width = "０１２３４５６７８９ＡＢＣＤＥＦＧＨＩＪＫＬＭＮＯＰＱＲＳＴＵＶＷＸＹＺａｂｃｄｅｆｇｈｉｊｋｌｍｎｏｐｑｒｓｔｕｖｗｘｙｚ！＃＄％＆’（）＊＋，－．／：；＜＝＞？＠［＼］＾＿｀｛｜｝～＂"
    mapping = str.maketrans(full_width, half_width)
    return text.translate(mapping)

class Vocabulary:
    def __init__(self,vocab_file):
        with open(vocab_file,'r',encoding='utf-8') as f:
            self.vocab = json.load(f)
        self.token_to_ids = {token: idx for idx, token in enumerate(self.vocab)}
        self.id_to_token = {idx: token for idx, token in enumerate(self.vocab)}
        self.pad_id = self.token_to_ids.get("<PAD>")
        self.unk_id = self.token_to_ids.get("<UNK>")
    def __len__(self):
        return len(self.vocab)
    def token_to_id(self,tokens):
        return list(self.token_to_ids.get(token , self.unk_id) for token in tokens)
    @classmethod
    def from_vocab_file(cls, vocab_file):
        return cls(vocab_file)

class BaseTokenizer(ABC):
    @abstractmethod
    def text_to_tokens(self, text):
        raise NotImplementedError
    @abstractmethod
    def tokens_to_ids(self, tokens):
        raise NotImplementedError
    
    def encoder(self, text):
        tokens = self.text_to_tokens(text)
        token_ids = self.tokens_to_ids(tokens)
        return token_ids
    @abstractmethod
    def get_pad_id(self):
        raise NotImplementedError
    
class CharTokenizer(BaseTokenizer):
    def __init__(self, vocab: Vocabulary):
        self.vocab = vocab
    def text_to_tokens(self, text):
        text = normalize_text(text)
        return list(text)
    def tokens_to_ids(self, tokens):
        return list(self.vocab.token_to_id(tokens))
    def get_pad_id(self):
        return self.vocab.pad_id

def transidx_to_tags(label_ids, id_to_tag):
    tags = []
    current_entity = None
    for i,label_id in enumerate(label_ids):
        tag = id_to_tag.get(label_id.item(), "O")
        if tag.startswith("B-"):
            current_entity = (tag[2:], i, i+1)
        elif tag.startswith("M-"):
            if current_entity and tag[2:] == current_entity[0]:
                current_entity = (current_entity[0], current_entity[1], i+1)
            else:
                current_entity = None
        elif tag.startswith("E-"):
            if current_entity and tag[2:] == current_entity[0]:
                current_entity = (current_entity[0], current_entity[1], i)
                tags.append(current_entity)
            current_entity = None
        elif tag.startswith("S-"):
            tags.append((tag[2:], i, i))
            current_entity = None
        else:
            current_entity = None
    return set(tags)

def caculate_metrics(pred_tags, true_tags, attention_mask, id_to_tag):
    true_entities = set()
    pred_entities = set()
    sample_idx = 0
    for pred, true, mask in zip(pred_tags,true_tags,attention_mask):
        B = true.shape[0]
        for i in range(B):
            row_mask = mask[i].bool()
            true_entity = true[i][row_mask]
            pred_entity = pred[i][row_mask]
            pe = transidx_to_tags(pred_entity, id_to_tag)
            te = transidx_to_tags(true_entity, id_to_tag)
            true_entities.update({(sample_idx,)+e for e in te})
            pred_entities.update({(sample_idx,)+e for e in pe})
            sample_idx += 1
    num_true = len(true_entities)
    num_pred = len(pred_entities)
    num_correct = len(true_entities.intersection(pred_entities))
    precision = num_correct / num_pred if num_pred > 0 else 0.0
    recall = num_correct / num_true if num_true > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}

class BaseNERNetwork(nn.Module, ABC):
    @abstractmethod
    def forward(self, token_ids, attention_mask):
        pass
config = NerConfig()
           
def main():
    seed_everything(getattr(config, 'seed', 42))
    vocab_path = os.path.join(config.data_dir, config.vocab_file)
    tag_map_path = os.path.join(config.data_dir, config.tag_map_file)
    vocab = Vocabulary.from_vocab_file(vocab_path)
    tag_map = load_json(tag_map_path)
    tokenizer = CharTokenizer(vocab)
    train_path = os.path.join(config.data_dir, config.train_file)
    dev_path = os.path.join(config.data_dir, config.dev_file)
    train_loader = create_ner_dataloader(train_path, tokenizer, tag_map, batch_size=config.batch_size, shuffle=True)
    dev_loader = create_ner_dataloader(dev_path, tokenizer, tag_map, batch_size=config.batch_size, shuffle=False)
    model = BiGRUNerNetWork(vocab_size=len(vocab), hidden_size=config.hidden_size, num_tags=len(tag_map), gru_layers=config.gru_layers)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
    if config.loss_type == "cross_entropy":
        loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
    else:
        loss_fn = NERLoss(loss_type=config.loss_type, entity_weight=config.entity_weight, hard_negative_ratio=config.hard_negative_ratio, ignore_index=-100)
    def eval_metric(all_logits, all_labels, all_attention_masks):
        all_pred_tags = [torch.argmax(logits, dim=-1) for logits in all_logits]
        all_labels_cpu = [labels.cpu() for labels in all_labels]
        all_attention_masks_cpu = [masks.cpu() for masks in all_attention_masks]
        all_preds_tags_cpu = [pred.cpu() for pred in all_pred_tags]
        active_masks = [mask.bool() for mask in all_attention_masks_cpu] 
        total_correct = 0
        total_pred = 0
        for pred_tags, true_labels, mask in zip(all_preds_tags_cpu, all_labels_cpu, active_masks):
            eq = (pred_tags == true_labels) & mask
            total_correct += eq.sum().item()
            total_pred += mask.sum().item()
        accuracy = total_correct / total_pred if total_pred > 0 else 0.0
        metric = caculate_metrics(all_pred_tags, all_labels_cpu, all_attention_masks_cpu, {v: k for k, v in tag_map.items()})
        metric['accuracy'] = accuracy
        return metric
    if config.resume_checkpoint and not os.path.exists(config.resume_checkpoint):
        print(f"Checkpoint file not found: {config.resume_checkpoint}. Starting training from scratch.")
    config.resume_checkpoint = None    
    trainer = Trainer(model=model, optimizer=optimizer, loss_fn=loss_fn, train_loader=train_loader, dev_loader=dev_loader, eval_metric=eval_metric, output_dir=config.output_dir, device=config.device, resume_checkpoint=config.resume_checkpoint, early_stopping_patience=config.early_stopping_patience)  
    os.makedirs(config.output_dir, exist_ok=True) 
    save_json(asdict(config), os.path.join(config.output_dir, "config.json"))
    print("Starting training with config:")
    trainer.fit(epochs=config.epochs)

if __name__ == "__main__":
    main()

Starting training with config:


Training: 100%|██████████| 469/469 [00:36<00:00, 12.80it/s]


Train Total Loss: 13.6658, NER Loss: 1.2336, Non-NER Loss: 1.3302


Evaluating: 100%|██████████| 157/157 [00:03<00:00, 44.26it/s]


Evaluation Metrics: {'precision': 0.6017508754377189, 'recall': 0.5371767963202787, 'f1': 0.5676332491800958, 'accuracy': 0.6906814290521676, 'loss': 10.2632079519284}
Validation Metrics: precision: 0.6018, recall: 0.5372, f1: 0.5676, accuracy: 0.6907, loss: 10.2632
New best model found! Saving model...


...

Evaluation Metrics: {'precision': 0.6478817342990376, 'recall': 0.5832179699013085, 'f1': 0.613851613358088, 'accuracy': 0.7645747123886442, 'loss': 14.566668401098555}
Validation Metrics: precision: 0.6479, recall: 0.5832, f1: 0.6139, accuracy: 0.7646, loss: 14.5667
EarlyStopping counter: 3 out of 5


Training: 100%|██████████| 469/469 [00:35<00:00, 13.14it/s]


Train Total Loss: 1.4016, NER Loss: 0.1079, Non-NER Loss: 0.3231


Evaluating: 100%|██████████| 157/157 [00:03<00:00, 40.50it/s]


Evaluation Metrics: {'precision': 0.6437017492426463, 'recall': 0.5883088465145357, 'f1': 0.6147600270654938, 'accuracy': 0.7656212442265095, 'loss': 16.353659881907664}
Validation Metrics: precision: 0.6437, recall: 0.5883, f1: 0.6148, accuracy: 0.7656, loss: 16.3537
EarlyStopping counter: 4 out of 5
Early stopping triggered. Stopping training.


In [13]:
from dataclasses import asdict
import os
import argparse
from abc import ABC, abstractmethod
import json
import torch.nn as nn
import torch
import torch.nn.utils.rnn as rnn
from torch.utils.data import Dataset,DataLoader
from torch.nn.utils.rnn import pad_sequence

def load_json(file_path):
    """从 JSON 文件加载数据。"""
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

def save_json(data, file_path):
    """将数据保存为 JSON 文件。"""
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

def normalize_text(text):
    half_width = r"0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz!#$%&'" + r'()*+,-./:;<=>?@[\]^_`{|}~".'
    full_width = "０１２３４５６７８９ＡＢＣＤＥＦＧＨＩＪＫＬＭＮＯＰＱＲＳＴＵＶＷＸＹＺａｂｃｄｅｆｇｈｉｊｋｌｍｎｏｐｑｒｓｔｕｖｗｘｙｚ！＃＄％＆’（）＊＋，－．／：；＜＝＞？＠［＼］＾＿｀｛｜｝～＂"
    mapping = str.maketrans(full_width, half_width)
    return text.translate(mapping)

class Vocabulary:
    def __init__(self,vocab_file):
        with open(vocab_file,'r',encoding='utf-8') as f:
            self.vocab = json.load(f)
        self.token_to_ids = {token: idx for idx, token in enumerate(self.vocab)}
        self.id_to_token = {idx: token for idx, token in enumerate(self.vocab)}
        self.pad_id = self.token_to_ids.get("<PAD>")
        self.unk_id = self.token_to_ids.get("<UNK>")
    def __len__(self):
        return len(self.vocab)
    def token_to_id(self,tokens):
        return list(self.token_to_ids.get(token , self.unk_id) for token in tokens)
    @classmethod
    def from_vocab_file(cls, vocab_file):
        return cls(vocab_file)

class BaseTokenizer(ABC):
    @abstractmethod
    def text_to_tokens(self, text):
        raise NotImplementedError
    @abstractmethod
    def tokens_to_ids(self, tokens):
        raise NotImplementedError
    
    def encoder(self, text):
        tokens = self.text_to_tokens(text)
        token_ids = self.tokens_to_ids(tokens)
        return token_ids
    @abstractmethod
    def get_pad_id(self):
        raise NotImplementedError
    
class CharTokenizer(BaseTokenizer):
    def __init__(self, vocab=Vocabulary):
        self.vocab = vocab
    def text_to_tokens(self, text):
        text = normalize_text(text)
        return list(text)
    def tokens_to_ids(self, tokens):
        return list(self.vocab.token_to_id(token) for token in tokens)
    def get_pad_id(self):
        return self.vocab.pad_id

class BiGRUNerNetWork(nn.Module):
    def __init__(self,vocab_size,hidden_size,num_tags,gru_layers=1):
        super(BiGRUNerNetWork,self).__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.gru = nn.ModuleList()
        for i in range(gru_layers):
            self.gru.append(nn.GRU(input_size=hidden_size, hidden_size=hidden_size, num_layers=1,batch_first=True,bidirectional=True))
        self.fc = nn.Linear(hidden_size * 2, hidden_size)
        self.classifier = nn.Linear(hidden_size, num_tags)
    def forward(self, token_ids, attention_mask):
        embeddings = self.embedding(token_ids)
        lengths = attention_mask.sum(dim=1).cpu().long()  
        packed_input = rnn.pack_padded_sequence(embeddings, lengths, batch_first=True, enforce_sorted=False)
        input = packed_input
        for gru_layer in self.gru:
            gru_output, _ = gru_layer(input)
            output,_ = rnn.pad_packed_sequence(gru_output,batch_first=True,total_length=token_ids.shape[1])##(batch_size, seq_len, hidden_size*2)
            feature = self.fc(output)
            pad_input,_ = rnn.pad_packed_sequence(input,batch_first=True,total_length=token_ids.shape[1])
            input = feature + pad_input
            input = rnn.pack_padded_sequence(input, lengths, batch_first=True, enforce_sorted=False)
        logits = self.classifier(feature)
        return logits
    
class NERPredictor:
    def __init__(self,model_dir,device="cpu"):
        self.device = device
        config_path = os.path.join(model_dir, "config.json")
        self.config = load_json(config_path)
        vocab_path = os.path.join(self.config['data_dir'], self.config['vocab_file'])
        tag_map_path = os.path.join(self.config['data_dir'], self.config['tag_map_file'])
        self.vocab = Vocabulary.from_vocab_file(vocab_path)
        self.tag_map = load_json(tag_map_path)
        self.tokenizer = CharTokenizer(self.vocab)
        self.model = BiGRUNerNetWork(vocab_size=len(self.vocab), hidden_size=self.config['hidden_size'], num_tags=len(self.tag_map), gru_layers=self.config['gru_layers'])
        model_path = os.path.join(model_dir, "best_model.pth")
        self.id_to_tag = {v: k for k, v in self.tag_map.items()}
        self.model.load_state_dict(torch.load(model_path, map_location=self.device)['model_state_dict'])
        self.model.to(self.device)
        self.model.eval()
        
    def predict(self, text):
        text = normalize_text(text)
        token_ids = torch.tensor(self.tokenizer.encoder(text), dtype=torch.long).unsqueeze(0).squeeze(2).to(self.device)
        print("token_ids shape:", token_ids.shape)
        attention_mask = (token_ids != self.tokenizer.get_pad_id()).long()
        with torch.no_grad():
            logits = self.model(token_ids, attention_mask)
        predictions = torch.argmax(logits, dim=-1).squeeze(0).cpu().tolist()
        tags = [self.id_to_tag.get(pred, "O") for pred in predictions]
        return self.extract_entities(list(text), tags)
        
    def extract_entities(self,tokens,tags):
        entities = []
        current_entity = None
        for i,tag in enumerate(tags):
            if tag.startswith("B-"):
                if current_entity:
                    pass
                current_entity = {"text":tokens[i],"type":tag[2:],"start":i}
            elif tag.startswith("M-"):
                if current_entity and tag[2:] == current_entity["type"]:
                    current_entity["text"] +=tokens[i]
                else:
                    current_entity = None
            elif tag.startswith("E-"):
                if current_entity and tag[2:] == current_entity["type"]:
                    current_entity["text"] +=tokens[i]
                    current_entity["end"] = i
                    entities.append(current_entity)
                    current_entity = None
            elif tag.startswith("S-"):
                entities.append({"text":tokens[i],"type":tag[2:],"start":i,"end":i})
                current_entity = None
            else:
                current_entity = None
        return entities    



def main():
    model_dir = "./model"
    text_to_predict = "患者自述发热、咳嗽，伴有轻微头痛前驱期持续1～4天，常有低热、头痛、乏力、咽痛、腹痛、烦躁等。"

    predictor = NERPredictor(model_dir=model_dir)
    entities = predictor.predict(text_to_predict)
    print(f"Text: {text_to_predict}")
    print(f"Entities: {json.dumps(entities, ensure_ascii=False, indent=2)}")

if __name__ == "__main__":
    main()
    

token_ids shape: torch.Size([1, 47])
Text: 患者自述发热、咳嗽，伴有轻微头痛前驱期持续1～4天，常有低热、头痛、乏力、咽痛、腹痛、烦躁等。
Entities: [
  {
    "text": "发热",
    "type": "sym",
    "start": 4,
    "end": 5
  },
  {
    "text": "咳嗽",
    "type": "sym",
    "start": 7,
    "end": 8
  },
  {
    "text": "头",
    "type": "bod",
    "start": 14,
    "end": 14
  },
  {
    "text": "低热",
    "type": "sym",
    "start": 28,
    "end": 29
  },
  {
    "text": "头",
    "type": "bod",
    "start": 31,
    "end": 31
  },
  {
    "text": "乏力",
    "type": "sym",
    "start": 34,
    "end": 35
  },
  {
    "text": "咽",
    "type": "bod",
    "start": 37,
    "end": 37
  },
  {
    "text": "腹",
    "type": "bod",
    "start": 40,
    "end": 40
  },
  {
    "text": "烦躁",
    "type": "sym",
    "start": 43,
    "end": 44
  }
]
